# **CIROH Code Repositories in GitHub** - Download and chunk code repositories information for CIROH AI Bot

In [100]:
import os
import io
import re
import json
import time
import zipfile
import requests
import shutil
import nbformat
import html
from pathlib import Path
from typing import Dict, Any, List, Optional
from urllib.parse import quote
from dotenv import load_dotenv
from openai import OpenAI
from database import DatabaseManager


In [86]:
# Environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Initialize OpenAI Client
client = OpenAI()

# Database Manager
db_manager = DatabaseManager()

### **Create GitHub Code Repositories: Download GitHub Repositories**

In [194]:
def authenticate_github() -> Dict[str, str]:
    """
    Returns headers for GitHub REST API.
    Works with public repos even without a token, but a token is strongly recommended
    to avoid rate limits.
    """
    token = os.getenv("GITHUB_TOKEN")

    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "CIROH-AI-Bot-Corpus-Builder"
    }

    if token:
        headers["Authorization"] = f"Bearer {token}"

    return headers

In [195]:
github_headers = authenticate_github()
print("GitHub authentication headers prepared.")
print("Using token:", "Authorization" in github_headers)

GitHub authentication headers prepared.
Using token: True


In [196]:
# =========================
# Configuration
# =========================

GITHUB_OWNER = "CIROH-UA"
BASE_CORPUS_PATH = Path("./GitHub-CIROH-CodeRepos")

# Repositories to exclude entirely from the GitHub corpus
EXCLUDED_REPOS = {
    "subdomain-hub",
    "ciroh_hub",
    "ciroh_hub_staging",
    "ciroh-portal",
    "ciroh-ua_website",
    "docuhub-staging",
}

OPTIONAL_EXCLUDED_REPOS = {
    ".github",
    "ciroh-ua.github.io",
    "subdomain-ngiab",
    "teehr-may-2023-workshop",
}

# Directories whose contents should never be downloaded into contents/
SKIP_DIR_PREFIXES = {
    ".git/",
    ".github/",
    ".ipynb_checkpoints/",
    "__pycache__/",
    "node_modules/",
    "dist/",
    "build/",
    "_build/",
    "site/",
    ".venv/",
    "venv/",
    "env/",
    ".mypy_cache/",
    ".pytest_cache/",
}

# Exact filenames that are considered useful inputs for all baselines
ALLOWED_EXACT_FILENAMES = {
    "readme",
    "readme.md",
    "readme.rst",
    "readme.txt",
    "license",
    "license.md",
    "license.txt",
    "contributing",
    "contributing.md",
    "contributing.rst",
    "contributing.txt",
    "changelog",
    "changelog.md",
    "changelog.rst",
    "changelog.txt",
    "citation",
    "citation.md",
    "citation.txt",
    "citation.cff",
    "security.md",
    "code_of_conduct.md",
    "requirements.txt",
    "environment.yml",
    "environment.yaml",
    "pyproject.toml",
    "setup.py",
    "setup.cfg",
    "pipfile",
    "pipfile.lock",
}

# Extensions that may be useful, but only under controlled conditions
ALLOWED_TEXT_EXTENSIONS = {
    ".md",
    ".markdown",
    ".rst",
    ".ipynb",
    ".cff",
    ".toml",
    ".yml",
    ".yaml",
    ".py",
}

# Path prefixes that strongly suggest human-readable documentation/tutorial content
ALLOWED_PATH_PREFIXES = {
    "docs/",
    "doc/",
    "documentation/",
    "examples/",
    "example/",
    "tutorials/",
    "tutorial/",
    "notebooks/",
    "notebook/",
    "workshop/",
    "workshops/",
}

# File name substrings that should be excluded even if the extension looks useful
EXCLUDED_NAME_SUBSTRINGS = {
    "checkpoint",
    "dummy",
    "placeholder",
    "deleteme",
    "tmp",
    "temp",
    "backup",
    "~",
}

EXCLUDED_RST_PATH_TOKENS = {
    "/api/",
    "/reference/",
    "/generated/",
    "_build/",
}

# Additional semantic folder signals beyond simple top-level prefixes.
# These help catch useful content in paths like:
# decision_trees/01.script/tutorial.ipynb
# neural_nets/lstm/01.script/example.py
SEMANTIC_PATH_TOKENS = {
    "script",
    "scripts",
    "notebook",
    "notebooks",
    "tutorial",
    "tutorials",
    "example",
    "examples",
    "demo",
    "demos",
    "exercise",
    "exercises",
    "lesson",
    "lessons",
    "lab",
    "labs",
    "training",
    "workflow",
    "workflows",
}

# Extensions allowed when a path contains semantic folder tokens
ALLOWED_SEMANTIC_FOLDER_EXTENSIONS = {
    ".ipynb",
    ".md",
    ".markdown",
    ".rst",
    ".txt",
    ".cff",
}

In [197]:
def github_get(url: str, headers: Dict[str, str], params: Optional[Dict[str, Any]] = None) -> requests.Response:
    resp = requests.get(url, headers=headers, params=params, timeout=60)
    resp.raise_for_status()
    return resp

In [198]:
def list_org_repositories(org_name: str, headers: Dict[str, str]) -> List[Dict[str, Any]]:
    """
    Lists all public repositories for a GitHub organization using pagination.
    """
    repos = []
    page = 1

    while True:
        resp = github_get(
            f"https://api.github.com/orgs/{org_name}/repos",
            headers=headers,
            params={
                "type": "public",
                "per_page": 100,
                "page": page,
                "sort": "full_name",
                "direction": "asc",
            }
        )
        batch = resp.json()
        if not batch:
            break

        repos.extend(batch)
        page += 1

    return repos

In [199]:
def build_repo_manifest_record(repo: Dict[str, Any], contributors_summary: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    license_info = repo.get("license") or {}

    return {
        "id": repo.get("id"),
        "name": repo.get("name"),
        "full_name": repo.get("full_name"),
        "html_url": repo.get("html_url"),
        "description": repo.get("description"),
        "homepage": repo.get("homepage"),
        "default_branch": repo.get("default_branch"),
        "created_at": repo.get("created_at"),
        "updated_at": repo.get("updated_at"),
        "pushed_at": repo.get("pushed_at"),
        "language": repo.get("language"),
        "topics": repo.get("topics", []),
        "fork": repo.get("fork", False),
        "archived": repo.get("archived", False),
        "disabled": repo.get("disabled", False),
        "visibility": repo.get("visibility"),
        "size_kb": repo.get("size"),
        "stargazers_count": repo.get("stargazers_count"),
        "watchers_count": repo.get("watchers_count"),
        "forks_count": repo.get("forks_count"),
        "open_issues_count": repo.get("open_issues_count"),
        "license": {
            "key": license_info.get("key"),
            "name": license_info.get("name"),
            "spdx_id": license_info.get("spdx_id"),
            "url": license_info.get("url"),
        } if license_info else None,
        "contributors_summary": contributors_summary or {
            "contributors_count": 0,
            "top_contributors": []
        }
    }

In [200]:
def get_repo_head_sha(owner: str, repo_name: str, branch: str, headers: Dict[str, str]) -> Optional[str]:
    """
    Returns the HEAD commit SHA for the default branch.
    Returns None when the repository is empty or has no accessible commits.
    """
    url = f"https://api.github.com/repos/{owner}/{repo_name}/commits/{branch}"

    try:
        resp = github_get(url, headers=headers)
        data = resp.json()
        return data["sha"]
    except requests.HTTPError as e:
        status = getattr(e.response, "status_code", None)
        if status == 409:
            print(f"  -> Warning: Repository '{owner}/{repo_name}' is empty or has no accessible commits on '{branch}'.")
            return None
        raise

In [201]:
def get_repo_contributors(owner: str, repo_name: str, headers: Dict[str, str]) -> List[Dict[str, Any]]:
    """
    Retrieves contributors for a repository.
    Returns a simplified list of contributors.
    """
    contributors = []
    page = 1

    while True:
        resp = github_get(
            f"https://api.github.com/repos/{owner}/{repo_name}/contributors",
            headers=headers,
            params={
                "per_page": 100,
                "page": page
            }
        )
        batch = resp.json()

        if not batch:
            break

        for c in batch:
            contributors.append({
                "id": c.get("id"),
                "login": c.get("login"),
                "html_url": c.get("html_url"),
                "type": c.get("type"),
                "contributions": c.get("contributions")
            })

        page += 1

    return contributors


def build_contributors_summary(contributors: List[Dict[str, Any]], top_k: int = 10) -> Dict[str, Any]:
    return {
        "contributors_count": len(contributors),
        "top_contributors": contributors[:top_k]
    }

In [202]:
def download_repo_snapshot_zip(owner: str, repo_name: str, commit_sha: str, headers: Dict[str, str]) -> bytes:
    """
    Downloads a zip archive snapshot for a specific commit SHA using the GitHub API.
    """
    url = f"https://api.github.com/repos/{owner}/{repo_name}/zipball/{commit_sha}"
    resp = github_get(url, headers=headers)
    return resp.content

In [203]:
def normalize_relative_path(relative_path: str) -> str:
    """
    Normalize a repository-relative path without accidentally stripping
    meaningful leading dots such as '.github/'.
    """
    rel = relative_path.replace("\\", "/").strip()

    if rel.startswith("./"):
        rel = rel[2:]

    return rel


def should_skip_by_path(relative_path: str) -> bool:
    rel = normalize_relative_path(relative_path)
    rel_lower = rel.lower()

    for prefix in SKIP_DIR_PREFIXES:
        if rel_lower.startswith(prefix):
            return True

    if "/.ipynb_checkpoints/" in rel_lower:
        return True

    return False


def normalize_path_segment(segment: str) -> str:
    """
    Normalize a path segment so folders like:
      01.script
      02_input
      03-output
    become semantically searchable as:
      script
      input
      output
    """
    s = segment.lower().strip()

    # remove leading numbering like 01., 02_, 03-, 04
    s = re.sub(r"^\d+[\._\- ]*", "", s)

    # replace separators with spaces
    s = re.sub(r"[\._\-]+", " ", s)

    # collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()

    return s


def path_has_semantic_folder_signal(relative_path: str) -> bool:
    """
    Returns True when any directory segment in the path appears semantically useful
    for tutorial/example/workflow content, even if the structure is not captured
    by ALLOWED_PATH_PREFIXES.
    """
    rel = normalize_relative_path(relative_path)
    parts = Path(rel).parts

    # check only directory segments, not the filename
    dir_parts = parts[:-1]

    for seg in dir_parts:
        norm = normalize_path_segment(seg)
        if not norm:
            continue

        tokens = set(norm.split())
        if tokens & SEMANTIC_PATH_TOKENS:
            return True

    return False


def classify_selection_reason(relative_path: str, repo_name: str) -> Optional[str]:
    """
    Returns a non-null reason if the file should be downloaded into contents/.
    Otherwise returns None.
    """
    rel = normalize_relative_path(relative_path)
    rel_lower = rel.lower()
    repo_name_lower = (repo_name or "").strip().lower()

    file_name = Path(rel_lower).name
    suffix = Path(file_name).suffix.lower()

    if should_skip_by_path(rel_lower):
        return None

    if any(token in file_name for token in EXCLUDED_NAME_SUBSTRINGS):
        return None

    # Exclude auto-generated / reference-style .rst files in noisy paths
    if suffix == ".rst" and any(token in rel_lower for token in EXCLUDED_RST_PATH_TOKENS):
        return None

    # Exclude repo autodoc stubs like hydrotools.something.rst
    if suffix == ".rst" and repo_name_lower and file_name.startswith(f"{repo_name_lower}."):
        return None

    # Exclude LICENSE.txt when it appears in build/vendor-like paths
    if file_name == "license.txt" and (
        "_build/" in rel_lower
        or "/vendor/" in rel_lower
        or "/vendors/" in rel_lower
        or "/third_party/" in rel_lower
        or "/third-party/" in rel_lower
        or "/site-packages/" in rel_lower
        or "/dist/" in rel_lower
        or "/build/" in rel_lower
    ):
        return None

    # 1. Always include high-value exact filenames
    if file_name in ALLOWED_EXACT_FILENAMES:
        return "allowed_exact_filename"

    # 2. Include documentation/tutorial/example paths only for selected useful extensions
    if any(rel_lower.startswith(prefix) for prefix in ALLOWED_PATH_PREFIXES):
        if suffix in ALLOWED_TEXT_EXTENSIONS:
            return "allowed_path_prefix"

    # 3. Include files located in semantically meaningful folders,
    #    but only for notebook/document-like files
    if path_has_semantic_folder_signal(rel_lower):
        if suffix in ALLOWED_SEMANTIC_FOLDER_EXTENSIONS:
            return "allowed_semantic_folder"

    # 4. Include top-level markdown/rst/cff docs
    if "/" not in rel_lower and suffix in {".md", ".markdown", ".rst", ".cff"}:
        return "allowed_top_level_doc"

    # 5. Include top-level notebooks
    if "/" not in rel_lower and suffix == ".ipynb":
        return "allowed_top_level_notebook"

    # 6. Do not include notebooks elsewhere unless already captured by a path rule above
    if suffix == ".ipynb":
        return None

    # 7. Include environment/build files only if explicitly whitelisted by exact filename
    if suffix in {".toml", ".yml", ".yaml", ".py"}:
        return None

    # 8. Never generally include .txt by extension alone
    if suffix == ".txt":
        return None

    return None

In [204]:
def extract_selected_files_from_zip(zip_bytes: bytes, output_dir: Path, repo_name: str) -> Dict[str, Any]:
    """
    Builds files_manifest.json for all files in the repository and extracts only
    selected files into contents/.
    Cleans previous extracted contents to avoid stale files from earlier runs.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    contents_dir = output_dir / "contents"

    # Clean previous extracted contents so the corpus reflects only the current selection policy
    if contents_dir.exists():
        shutil.rmtree(contents_dir)
    contents_dir.mkdir(parents=True, exist_ok=True)

    files_manifest = []

    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        members = [m for m in zf.infolist() if not m.is_dir()]

        for member in members:
            parts = Path(member.filename).parts
            if len(parts) < 2:
                continue

            relative_path = str(Path(*parts[1:]))
            relative_path_norm = relative_path.replace("\\", "/")

            selection_reason = classify_selection_reason(relative_path_norm, repo_name=repo_name)
            downloaded = selection_reason is not None

            if downloaded:
                target_path = contents_dir / relative_path_norm
                target_path.parent.mkdir(parents=True, exist_ok=True)

                with zf.open(member, "r") as src, open(target_path, "wb") as dst:
                    dst.write(src.read())

            files_manifest.append({
                "path": relative_path_norm,
                "file_name": Path(relative_path_norm).name,
                "extension": Path(relative_path_norm).suffix.lower(),
                "size_bytes": member.file_size,
                "downloaded": downloaded,
                "selection_reason": selection_reason,
            })

    downloaded_files_count = sum(1 for x in files_manifest if x["downloaded"])

    return {
        "files_manifest": files_manifest,
        "downloaded_files_count": downloaded_files_count,
    }

In [205]:
def download_github_corpus(
    repos: List[Dict[str, Any]],
    headers: Dict[str, str],
    base_path: Path = BASE_CORPUS_PATH
) -> Dict[str, Any]:
    """
    Downloads the GitHub Level-0 corpus for selected repositories.
    Saves:
      - repo_metadata.json
      - archive_info.json
      - files_manifest.json
      - contributors.json
      - contents/ (selected files only)
    """
    base_path.mkdir(parents=True, exist_ok=True)

    success_count = 0
    failed_repos = []
    skipped_repos = []
    report = []
    manifest_records = []

    for index, repo in enumerate(repos, start=1):
        repo_name = repo["name"]
        owner = repo["owner"]["login"]
        default_branch = repo["default_branch"]

        print(f"[{index}/{len(repos)}] Processing repository: {owner}/{repo_name}")

        repo_dir = base_path / repo_name
        repo_dir.mkdir(parents=True, exist_ok=True)

        try:
            head_sha = get_repo_head_sha(owner, repo_name, default_branch, headers)

            if not head_sha:
                print(f"  -> Skipped: empty repository or no accessible HEAD commit")
                skipped_repos.append(repo_name)
                report.append({
                    "repo_name": repo_name,
                    "full_name": repo.get("full_name"),
                    "status": "skipped",
                    "reason": "empty_repository"
                })
                continue

            contributors = get_repo_contributors(owner, repo_name, headers)
            contributors_summary = build_contributors_summary(contributors)

            metadata = build_repo_manifest_record(repo, contributors_summary=contributors_summary)

            archive_info = {
                "owner": owner,
                "repo_name": repo_name,
                "default_branch": default_branch,
                "frozen_commit_sha": head_sha,
                "downloaded_at_epoch": time.time(),
                "archive_format": "zip",
            }

            zip_bytes = download_repo_snapshot_zip(owner, repo_name, head_sha, headers)
            extracted = extract_selected_files_from_zip(zip_bytes, repo_dir, repo_name=repo_name)

            with open(repo_dir / "repo_metadata.json", "w", encoding="utf-8") as f:
                json.dump(metadata, f, indent=4, ensure_ascii=False)

            with open(repo_dir / "archive_info.json", "w", encoding="utf-8") as f:
                json.dump(archive_info, f, indent=4, ensure_ascii=False)

            with open(repo_dir / "files_manifest.json", "w", encoding="utf-8") as f:
                json.dump(extracted["files_manifest"], f, indent=4, ensure_ascii=False)

            with open(repo_dir / "contributors.json", "w", encoding="utf-8") as f:
                json.dump(contributors, f, indent=4, ensure_ascii=False)

            report.append({
                "repo_name": repo_name,
                "full_name": repo["full_name"],
                "frozen_commit_sha": head_sha,
                "total_files": len(extracted["files_manifest"]),
                "downloaded_files": extracted["downloaded_files_count"],
                "contributors_count": len(contributors),
                "status": "success",
            })

            manifest_records.append(metadata)
            success_count += 1

        except Exception as e:
            print(f"  -> Failure: {e}")
            failed_repos.append(repo_name)
            report.append({
                "repo_name": repo_name,
                "full_name": repo.get("full_name"),
                "status": "failed",
                "error": str(e)
            })

    with open(base_path / "repo_download_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=4, ensure_ascii=False)

    with open(base_path / "github_org_manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest_records, f, indent=4, ensure_ascii=False)

    return {
        "total_processed": len(repos),
        "successful_downloads": success_count,
        "failed_downloads": len(failed_repos),
        "skipped_downloads": len(skipped_repos),
        "failed_repos": failed_repos,
        "skipped_repos": skipped_repos,
    }

In [206]:
all_repos = list_org_repositories(GITHUB_OWNER, github_headers)
print(f"Repositories found in organization: {len(all_repos)}")

selected_repos = []
for repo in all_repos:
    name = repo.get("name")
    if name in EXCLUDED_REPOS:
        continue
    if name in OPTIONAL_EXCLUDED_REPOS:
        continue
    selected_repos.append(repo)

print(f"Repositories selected for download: {len(selected_repos)}")
print("Sample selected repos:", [r["name"] for r in selected_repos[:10]])

Repositories found in organization: 61
Repositories selected for download: 51
Sample selected repos: ['api-nwm-gcp', 'awi-ciroh-image', 'CAMELS_data_sample', 'cfe_v1.0', 'CIROH-open-source-project-template', 'ciroh_pyngiab', 'Community-Streamflow-Evaluation-System', 'community_hf_patcher', 'Conferences', 'datastreamcli']


In [207]:
report = download_github_corpus(
    repos=selected_repos,
    headers=github_headers,
    base_path=BASE_CORPUS_PATH
)

print("\n" + "=" * 50)
print("GITHUB CORPUS ACQUISITION REPORT")
print("=" * 50)
print(f"Total processed: {report['total_processed']}")
print(f"Successful downloads: {report['successful_downloads']}")
print(f"Skipped downloads: {report['skipped_downloads']}")
print(f"Failed downloads: {report['failed_downloads']}")

if report["skipped_repos"]:
    print("\nSkipped repositories:")
    for repo_name in report["skipped_repos"]:
        print(" -", repo_name)

if report["failed_repos"]:
    print("\nFailed repositories:")
    for repo_name in report["failed_repos"]:
        print(" -", repo_name)

[1/51] Processing repository: CIROH-UA/api-nwm-gcp
[2/51] Processing repository: CIROH-UA/awi-ciroh-image
[3/51] Processing repository: CIROH-UA/CAMELS_data_sample
[4/51] Processing repository: CIROH-UA/cfe_v1.0
[5/51] Processing repository: CIROH-UA/CIROH-open-source-project-template
[6/51] Processing repository: CIROH-UA/ciroh_pyngiab
[7/51] Processing repository: CIROH-UA/Community-Streamflow-Evaluation-System
[8/51] Processing repository: CIROH-UA/community_hf_patcher
[9/51] Processing repository: CIROH-UA/Conferences
[10/51] Processing repository: CIROH-UA/datastreamcli
[11/51] Processing repository: CIROH-UA/data_access_example
[12/51] Processing repository: CIROH-UA/deep_bucket_lab
[13/51] Processing repository: CIROH-UA/DEVCON_SNOW_ML
[14/51] Processing repository: CIROH-UA/forcingprocessor
[15/51] Processing repository: CIROH-UA/forecast_data_overlay
[16/51] Processing repository: CIROH-UA/GEE_Workshop
[17/51] Processing repository: CIROH-UA/hf_pmtiles
[18/51] Processing repos

### **Create GitHub Code Repositories: Artifacts**

In [208]:
# =========================
# Configuration
# =========================

BASE_GITHUB_CORPUS_DIR = Path("./GitHub-CIROH-CodeRepos")
OUTPUT_PATH = Path("json/coderepo_artifacts.json")

CODEREPO_ARTIFACT_TYPE_ID = 4  # TBLArtifactTypes -> Code Repository

SUMMARY_MODEL = "gpt-5.5"
MAX_REPO_CONTEXT_CHARS = 350_000
MAX_TEXT_FILE_CHARS = 25_000
MAX_NOTEBOOK_MARKDOWN_CHARS = 35_000
MAX_CONFIG_FILE_CHARS = 12_000
MAX_FILES_IN_SUMMARY_CONTEXT = 80

In [209]:
# =========================
# Helpers
# =========================


def read_json(path: Path, default: Any = None) -> Any:
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def clean_text(value: Any) -> Optional[str]:
    if value is None:
        return None
    if not isinstance(value, str):
        value = str(value)
    value = value.replace("\x00", "").strip()
    return value if value else None


def normalize_list(value: Any) -> List[Any]:
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return []


def summarize_selected_files(files_manifest: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Summarize the selected/downloaded files so the artifact keeps high-level
    provenance without duplicating chunk-level content.
    """
    downloaded = [f for f in files_manifest if isinstance(f, dict) and f.get("downloaded", False)]

    extensions_counter = {}
    selection_reason_counter = {}
    sample_paths = []

    for rec in downloaded:
        ext = clean_text(rec.get("extension")) or ""
        reason = clean_text(rec.get("selection_reason")) or ""

        extensions_counter[ext] = extensions_counter.get(ext, 0) + 1
        selection_reason_counter[reason] = selection_reason_counter.get(reason, 0) + 1

        path_value = clean_text(rec.get("path"))
        if path_value and len(sample_paths) < 20:
            sample_paths.append(path_value)

    return {
        "downloaded_files_count": len(downloaded),
        "downloaded_extensions": dict(sorted(extensions_counter.items(), key=lambda x: x[0])),
        "selection_reason_counts": dict(sorted(selection_reason_counter.items(), key=lambda x: x[0])),
        "sample_downloaded_paths": sample_paths
    }


def sanitize_text(s: str) -> str:
    if s is None:
        return ""
    return str(s).replace("\x00", "").strip()


def truncate_text(text: str, max_chars: int) -> str:
    text = sanitize_text(text)
    if len(text) <= max_chars:
        return text
    head = text[: int(max_chars * 0.7)]
    tail = text[-int(max_chars * 0.3):]
    return head + "\n\n...\n\n" + tail


def get_downloaded_files(files_manifest: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    downloaded = [
        rec for rec in files_manifest
        if isinstance(rec, dict) and rec.get("downloaded", False)
    ]
    downloaded.sort(key=lambda x: (str(x.get("path", "")).lower()))
    return downloaded


def file_priority(rec: Dict[str, Any]) -> tuple:
    """
    Lower tuple = higher priority in the summary context.
    """
    path_value = (rec.get("path") or "").replace("\\", "/").lower()
    file_name = Path(path_value).name
    ext = Path(path_value).suffix.lower()

    if file_name.startswith("readme"):
        return (0, path_value)
    if path_value in {"docs/index.rst", "docs/index.md", "docs/index.markdown"}:
        return (1, path_value)
    if ext == ".ipynb":
        return (2, path_value)
    if path_value.startswith(("docs/", "doc/", "documentation/", "tutorials/", "tutorial/", "examples/", "example/", "workshops/", "workshop/")):
        return (3, path_value)
    if file_name in {"requirements.txt", "environment.yml", "environment.yaml", "pyproject.toml", "setup.py", "setup.cfg", "pipfile", "pipfile.lock"}:
        return (4, path_value)
    if file_name.startswith(("contributing", "changelog", "citation", "license", "security")):
        return (5, path_value)
    return (9, path_value)


def load_text_file(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8", errors="replace")
    except Exception:
        return ""


def extract_notebook_markdown(path: Path) -> str:
    """
    Extract only markdown cells from a notebook for repository-level summarization.
    """
    try:
        nb = nbformat.read(path, as_version=4)
    except Exception:
        return ""

    markdown_blocks = []
    for cell in nb.cells:
        if cell.get("cell_type") == "markdown":
            src = "".join(cell.get("source", ""))
            src = sanitize_text(src)
            if src:
                markdown_blocks.append(src)

    return "\n\n".join(markdown_blocks)


def summarize_config_file(path: Path) -> str:
    """
    Keep configuration files compact. We want signal, not full raw content.
    """
    text = load_text_file(path)
    text = sanitize_text(text)
    if not text:
        return ""

    file_name = path.name.lower()

    if file_name == "requirements.txt":
        lines = [
            line.strip() for line in text.splitlines()
            if line.strip() and not line.strip().startswith("#")
        ]
        return "Python dependencies declared in requirements.txt:\n" + "\n".join(lines[:200])

    if file_name in {"environment.yml", "environment.yaml"}:
        return truncate_text("Environment configuration:\n" + text, MAX_CONFIG_FILE_CHARS)

    if file_name in {"pyproject.toml", "setup.cfg", "setup.py", "pipfile", "pipfile.lock"}:
        return truncate_text(f"Project/package configuration from {path.name}:\n{text}", MAX_CONFIG_FILE_CHARS)

    return truncate_text(text, MAX_CONFIG_FILE_CHARS)


def read_selected_file_content(repo_dir: Path, file_rec: Dict[str, Any]) -> str:
    relative_path = clean_text(file_rec.get("path"))
    if not relative_path:
        return ""

    file_path = repo_dir / "contents" / relative_path
    if not file_path.exists():
        return ""

    suffix = file_path.suffix.lower()

    if suffix == ".ipynb":
        return truncate_text(extract_notebook_markdown(file_path), MAX_NOTEBOOK_MARKDOWN_CHARS)

    if file_path.name.lower() in {
        "requirements.txt", "environment.yml", "environment.yaml",
        "pyproject.toml", "setup.py", "setup.cfg", "pipfile", "pipfile.lock"
    }:
        return summarize_config_file(file_path)

    if suffix in {".md", ".markdown", ".rst", ".txt", ".cff", ".py", ".toml", ".yml", ".yaml"}:
        return truncate_text(load_text_file(file_path), MAX_TEXT_FILE_CHARS)

    return ""


def build_repo_summary_input(
    repo_dir: Path,
    repo_name: str,
    repo_metadata: Dict[str, Any],
    files_manifest: List[Dict[str, Any]]
) -> str:
    """
    Build a compact but information-dense text dossier for one repository.
    This is the input that will be summarized by the LLM.
    """
    downloaded = get_downloaded_files(files_manifest)
    downloaded = sorted(downloaded, key=file_priority)[:MAX_FILES_IN_SUMMARY_CONTEXT]

    header_parts = [
        f"Repository name: {repo_name}",
        f"Repository full name: {clean_text(repo_metadata.get('full_name')) or ''}",
        f"Description: {clean_text(repo_metadata.get('description')) or ''}",
        f"Primary language: {clean_text(repo_metadata.get('language')) or ''}",
        f"Topics: {', '.join(normalize_list(repo_metadata.get('topics')))}",
        f"Default branch: {clean_text(repo_metadata.get('default_branch')) or ''}",
    ]

    sections = ["\n".join([x for x in header_parts if x.strip()])]

    for rec in downloaded:
        rel_path = clean_text(rec.get("path")) or ""
        selection_reason = clean_text(rec.get("selection_reason")) or ""
        ext = clean_text(rec.get("extension")) or ""
        content = read_selected_file_content(repo_dir, rec)

        if not content:
            continue

        section = (
            f"\n--- FILE START ---\n"
            f"Path: {rel_path}\n"
            f"Extension: {ext}\n"
            f"Selection reason: {selection_reason}\n"
            f"Content:\n{content}\n"
            f"--- FILE END ---"
        )
        sections.append(section)

    full_context = "\n\n".join(sections)
    return truncate_text(full_context, MAX_REPO_CONTEXT_CHARS)


SYSTEM_MSG = (
    "You are a precise CIROH code repository summarizer. "
    "Return ONLY valid JSON. No prose, no markdown."
)


def build_coderepo_prompt(repo_context: str) -> str:
    return f"""
# ROLE & GOAL
You are an AI Knowledge Architect specializing in hydrology, scientific computing, and research software for the Cooperative Institute for Research to Operations in Hydrology (CIROH). Your task is to summarize a GitHub code repository using an aggregated textual representation built from the repository metadata and selected files.

# CONTEXT
The output will populate a JSONB field called `summary_data` in a PostgreSQL database used by a Retrieval-Augmented Generation (RAG) system. The `summary_text` will later be embedded for semantic search. Therefore, the summary must help retrieve repositories relevant to user questions about hydrologic models, tools, workflows, tutorials, infrastructure, data access, scientific software, evaluation pipelines, deployment, and training materials.

# REPOSITORY CONTENT TO ANALYZE
---
**[BEGINNING OF REPOSITORY DOSSIER]**

{repo_context}

**[END OF REPOSITORY DOSSIER]**
---

# INSTRUCTIONS
Based on the content provided above, generate a single JSON object.

1. The `summary_text` must describe the repository as a whole, not just one file.
2. Infer the repository's main purpose from the combined evidence across README files, notebooks, docs, and configuration files.
3. Prioritize the repository’s central purpose, core workflows, and major technical components. Do not let low-level implementation details from individual scripts or endpoint parameters dominate the summary.
4. The `keywords` array should capture major technical themes, workflows, tools, models, or capabilities.
5. The `entities` array should include specific named tools, frameworks, libraries, systems, hydrologic models, or platforms explicitly referenced in the repository content.
6. The `document_type` should be a concise characterization chosen from the repository evidence, such as "tutorial repository", "scientific software package", "deployment repository", "training materials repository", "data access utilities repository", "hydrologic modeling repository", etc.
7. Do not invent capabilities not supported by the repository content.
8. Be direct, factual, and information-dense.

# REQUIRED OUTPUT FORMAT
Generate a single valid JSON object with exactly this structure:

{{
  "summary_text": "",
  "keywords": [],
  "entities": [],
  "document_type": ""
}}
""".strip()


def summarize_coderepo_with_llm(repo_context: str) -> Dict[str, Any]:
    prompt = build_coderepo_prompt(repo_context)

    #print(f"PROMPT:{'='*40}\n{prompt}\n{'='*40}\n")

    resp = client.responses.create(
        model=SUMMARY_MODEL,
        input=[
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": prompt},
        ],
        text={
            "format": {"type": "json_object"},
            "verbosity": "medium"
        },
        reasoning={"effort": "low"},
    )

    raw = (resp.output_text or "").strip()
    data = json.loads(raw)

    return {
        "summary_text": data.get("summary_text") or "",
        "keywords": data.get("keywords") if isinstance(data.get("keywords"), list) else [],
        "entities": data.get("entities") if isinstance(data.get("entities"), list) else [],
        "document_type": data.get("document_type") or "",
    }


def build_coderepo_summary_data(repo_dir: Path, repo_metadata: Dict[str, Any], files_manifest: List[Dict[str, Any]]) -> Dict[str, Any]:
    repo_name = clean_text(repo_metadata.get("name")) or repo_dir.name
    repo_context = build_repo_summary_input(
        repo_dir=repo_dir,
        repo_name=repo_name,
        repo_metadata=repo_metadata,
        files_manifest=files_manifest
    )
    return summarize_coderepo_with_llm(repo_context)

In [210]:
# =========================
# Build one artifact per repository
# =========================

def build_coderepo_artifact_record(repo_dir: Path, id_artifact: int) -> Dict[str, Any]:
    repo_metadata = read_json(repo_dir / "repo_metadata.json", {})
    archive_info = read_json(repo_dir / "archive_info.json", {})
    files_manifest = read_json(repo_dir / "files_manifest.json", [])
    contributors = read_json(repo_dir / "contributors.json", [])

    repo_name = clean_text(repo_metadata.get("name")) or repo_dir.name
    full_name = clean_text(repo_metadata.get("full_name")) or f"CIROH-UA/{repo_name}"
    repo_url = clean_text(repo_metadata.get("html_url")) or f"https://github.com/{full_name}"

    summary_data = build_coderepo_summary_data(
        repo_dir=repo_dir,
        repo_metadata=repo_metadata,
        files_manifest=files_manifest
    )

    return {
        "idArtifact": id_artifact,
        "idArtifactType": CODEREPO_ARTIFACT_TYPE_ID,
        "Title": repo_name,
        "URL": repo_url,
        "idArtifactParent": None,

        # High-level repository description
        "description": clean_text(repo_metadata.get("description")),
        "topics": normalize_list(repo_metadata.get("topics")),
        "default_branch": clean_text(repo_metadata.get("default_branch")),
        "frozen_commit_sha": clean_text(archive_info.get("frozen_commit_sha")),
        "language": clean_text(repo_metadata.get("language")),

        # Repository timestamps
        "created_at": clean_text(repo_metadata.get("created_at")),
        "updated_at": clean_text(repo_metadata.get("updated_at")),
        "pushed_at": clean_text(repo_metadata.get("pushed_at")),

        # Contributor provenance
        "contributors_summary": repo_metadata.get("contributors_summary"),

        # Snapshot / corpus provenance
        "full_name": full_name,
        "artifact_role": "repository",
        "selected_files_summary": summarize_selected_files(files_manifest),

        # RAG summary payload
        "summary_data": summary_data
    }

In [211]:
# =========================
# Build coderepo_artifacts.json
# =========================

if not BASE_GITHUB_CORPUS_DIR.exists():
    raise FileNotFoundError(f"Base directory not found: {BASE_GITHUB_CORPUS_DIR}")

repo_dirs = sorted(
    [p for p in BASE_GITHUB_CORPUS_DIR.iterdir() if p.is_dir()],
    key=lambda p: p.name.lower()
)

print(f"Repository folders detected: {len(repo_dirs)}")

coderepo_artifacts = []
errors = []

for idx, repo_dir in enumerate(repo_dirs, start=1):
    try:
        print(f"[{idx}/{len(repo_dirs)}] Processing repository: {repo_dir.name}")
        rec = build_coderepo_artifact_record(repo_dir, id_artifact=idx)
        coderepo_artifacts.append(rec)
    except Exception as e:
        errors.append({
            "repo_name": repo_dir.name,
            "error": str(e)
        })

print(f"Artifacts built: {len(coderepo_artifacts)}")
print(f"Errors: {len(errors)}")

if errors:
    print("Error samples:")
    for e in errors[:10]:
        print(e)

Repository folders detected: 51
[1/51] Processing repository: api-nwm-gcp
[2/51] Processing repository: awi-ciroh-image
[3/51] Processing repository: CAMELS_data_sample
[4/51] Processing repository: cfe_v1.0
[5/51] Processing repository: CIROH-open-source-project-template
[6/51] Processing repository: ciroh_pyngiab
[7/51] Processing repository: Community-Streamflow-Evaluation-System
[8/51] Processing repository: community_hf_patcher
[9/51] Processing repository: Conferences
[10/51] Processing repository: data_access_example
[11/51] Processing repository: datastreamcli
[12/51] Processing repository: deep_bucket_lab
[13/51] Processing repository: DEVCON_SNOW_ML
[14/51] Processing repository: forcingprocessor
[15/51] Processing repository: forecast_data_overlay
[16/51] Processing repository: GEE_Workshop
[17/51] Processing repository: hf_pmtiles
[18/51] Processing repository: hydromachine-tutorials
[19/51] Processing repository: hydrotools
[20/51] Processing repository: lstm
[21/51] Proce

In [212]:
# =========================
# Persist file
# =========================

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(coderepo_artifacts, f, indent=2, ensure_ascii=False)

print(f"File saved to: {OUTPUT_PATH}")
print(f"Total records: {len(coderepo_artifacts)}")

File saved to: json/coderepo_artifacts.json
Total records: 51


In [213]:
# =========================
# Quick inspection
# =========================

if coderepo_artifacts:
    print(json.dumps(coderepo_artifacts[:5], indent=2, ensure_ascii=False))

[
  {
    "idArtifact": 1,
    "idArtifactType": 4,
    "Title": "api-nwm-gcp",
    "URL": "https://github.com/CIROH-UA/api-nwm-gcp",
    "idArtifactParent": null,
    "description": "REST API backed by National Water Model data, developed on Google Cloud Platform",
    "topics": [],
    "default_branch": "main",
    "frozen_commit_sha": "3e445c6ba0043b5c2625e75813285603639cd932",
    "language": "Jupyter Notebook",
    "created_at": "2025-01-24T16:16:41Z",
    "updated_at": "2025-08-19T20:44:11Z",
    "pushed_at": "2025-03-06T04:24:02Z",
    "contributors_summary": {
      "contributors_count": 5,
      "top_contributors": [
        {
          "id": 11776290,
          "login": "KMarkert",
          "html_url": "https://github.com/KMarkert",
          "type": "User",
          "contributions": 39
        },
        {
          "id": 102833747,
          "login": "gmds2",
          "html_url": "https://github.com/gmds2",
          "type": "User",
          "contributions": 31
        

### **Create GitHub Code Repositories: Chunks**

In [101]:
# =========================
# Configuration
# =========================

BASE_CODE_REPOS_DIR = Path("./GitHub-CIROH-CodeRepos")
ARTIFACTS_PATH = Path("json/coderepo_artifacts.json")
OUTPUT_CHUNKS_PATH = Path("json/coderepo_chunks.json")

CODEREPO_ARTIFACT_TYPE_ID = 4
SUMMARY_MODEL = "gpt-5.5"

MAX_TEXT_CHARS_PER_CHUNK = 12000
MIN_TEXT_LENGTH = 40

In [102]:
# =========================
# Resolve code-repository chunk types from DB
# =========================

with db_manager as db:
    rows = db.execute_query(
        """
        SELECT idchunktype, typename
        FROM tblchunktypes
        WHERE idartifacttype = %s;
        """,
        (CODEREPO_ARTIFACT_TYPE_ID,),
        fetch=True
    ) or []

if not rows:
    raise RuntimeError(
        f"No chunk types found in TBLChunkTypes for idArtifactType={CODEREPO_ARTIFACT_TYPE_ID}"
    )

chunk_type_name_to_id = {
    r["typename"]: r["idchunktype"] for r in rows
}

required_chunk_names = {
    "Project Overview",
    "Installation Setup",
    "Usage Examples",
    "Repository Structure",
    "Contributing Guidelines",
    "License Citation",
    "Documentation Section",
    "Notebook Section",
    "Configuration / Deployment",
    "Change Log / Release Notes",
}

missing_names = required_chunk_names - set(chunk_type_name_to_id.keys())
if missing_names:
    raise RuntimeError(
        "Missing required code-repository chunk types in DB: " + ", ".join(sorted(missing_names))
    )

print("Loaded code-repository chunk types from DB:")
for name, chunk_id in sorted(chunk_type_name_to_id.items(), key=lambda x: x[1]):
    print(f" - {name}: {chunk_id}")

Loaded code-repository chunk types from DB:
 - Project Overview: 21
 - Installation Setup: 22
 - Usage Examples: 23
 - Repository Structure: 24
 - Contributing Guidelines: 25
 - License Citation: 26
 - Documentation Section: 48
 - Notebook Section: 49
 - Configuration / Deployment: 50
 - Change Log / Release Notes: 51


In [103]:
# =========================
# Load coderepo_artifacts.json and build mappings
# =========================

with open(ARTIFACTS_PATH, "r", encoding="utf-8") as f:
    coderepo_artifacts = json.load(f)

repo_name_to_artifact = {}
repo_full_name_to_artifact = {}

for rec in coderepo_artifacts:
    repo_name = rec.get("Title")
    full_name = rec.get("full_name")
    if repo_name:
        repo_name_to_artifact[str(repo_name)] = rec
    if full_name:
        repo_full_name_to_artifact[str(full_name)] = rec

print(f"Loaded code repository artifacts: {len(coderepo_artifacts)}")
print(f"Mapped repo names: {len(repo_name_to_artifact)}")

Loaded code repository artifacts: 51
Mapped repo names: 51


In [104]:
# =========================
# Generic helpers
# =========================

MD_HEADING_RE = re.compile(r"^(#{1,6})\s+(.*\S)\s*$")
MD_RULE_RE = re.compile(r"^\s*([-*_])\1{2,}\s*$")

RST_UNDERLINE_CHARS = set("=-~^\"`:+*#")
URL_RE = re.compile(r"https?://[^\s<>\"]+")

CONFIG_FILE_NAMES = {
    "requirements.txt",
    "environment.yml",
    "environment.yaml",
    "pyproject.toml",
    "setup.py",
    "setup.cfg",
    "pipfile",
    "pipfile.lock",
    "dockerfile",
    "docker-compose.yml",
    "docker-compose.yaml",
}

CONTRIB_FILE_NAMES = {
    "contributing",
    "contributing.md",
    "contributing.rst",
    "contributing.txt",
    "code_of_conduct.md",
}

CHANGELOG_FILE_NAMES = {
    "changelog",
    "changelog.md",
    "changelog.rst",
    "changelog.txt",
    "release-notes.md",
    "release_notes.md",
}

LICENSE_FILE_NAMES = {
    "license",
    "license.md",
    "license.txt",
    "citation",
    "citation.md",
    "citation.txt",
    "citation.cff",
    "terms.md",
    "terms.txt",
}

OVERVIEW_HEADING_HINTS = {
    "overview", "introduction", "about", "purpose", "motivation", "summary"
}

INSTALL_HEADING_HINTS = {
    "installation", "install", "setup", "getting started", "build and run", "build", "prepare the python environment"
}

USAGE_HEADING_HINTS = {
    "usage", "example", "examples", "quick start", "run", "workflow", "workflows", "tutorial", "steps", "how to run"
}

STRUCTURE_HEADING_HINTS = {
    "repository structure", "project structure", "contents", "what's here", "content of this resource", "folder structure"
}


def clean_text(value: Any) -> Optional[str]:
    if value is None:
        return None
    if not isinstance(value, str):
        value = str(value)
    value = value.replace("\x00", "")
    value = html.unescape(value)
    value = value.strip()
    return value or None


def clean_multiline_text(value: Any) -> str:
    value = clean_text(value) or ""
    value = value.replace("\r\n", "\n").replace("\r", "\n")
    value = re.sub(r"\n{3,}", "\n\n", value)
    return value.strip()


def read_json(path: Path, default: Any = None) -> Any:
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def truncate_text(text: str, max_chars: int = MAX_TEXT_CHARS_PER_CHUNK) -> str:
    text = clean_multiline_text(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "\n\n[TRUNCATED]"


def normalize_heading(text: Optional[str]) -> Optional[str]:
    t = clean_text(text)
    if not t:
        return None
    t = t.lower()
    t = re.sub(r"\s+", " ", t)
    return t.strip()


def build_blob_url(full_name: str, frozen_commit_sha: str, relative_path: str) -> str:
    safe_path = quote(relative_path.replace("\\", "/"), safe="/")
    return f"https://github.com/{full_name}/blob/{frozen_commit_sha}/{safe_path}"

In [105]:
# =========================
# Heuristic chunk-type assignment
# =========================

def classify_obvious_chunk_type(
    *,
    source_path: str,
    source_format: str,
    heading: Optional[str],
) -> Optional[str]:
    """
    Deterministic assignment for obvious cases.
    Returns a chunk type name or None when ambiguous.
    """
    path_lower = (source_path or "").lower().replace("\\", "/")
    file_name = Path(path_lower).name
    heading_norm = normalize_heading(heading)

    # 1. Notebook-origin chunks are always Notebook Section
    if source_format == "notebook":
        return "Notebook Section"

    # 2. File-name based obvious cases
    if file_name in CONTRIB_FILE_NAMES:
        return "Contributing Guidelines"

    if file_name in CHANGELOG_FILE_NAMES:
        return "Change Log / Release Notes"

    if file_name in LICENSE_FILE_NAMES:
        return "License Citation"

    if file_name in CONFIG_FILE_NAMES:
        return "Configuration / Deployment"

    # 3. Path-based configuration/deployment hints
    if any(tok in path_lower for tok in [
        "/deploy", "/deployment", "/docker", "/terraform", "/config", "/configs/",
        "/workflow", "/workflows", ".github/workflows/"
    ]):
        return "Configuration / Deployment"

    # 4. Heading-based strong hints
    if heading_norm:
        if heading_norm in OVERVIEW_HEADING_HINTS:
            return "Project Overview"
        if heading_norm in INSTALL_HEADING_HINTS:
            return "Installation Setup"
        if heading_norm in USAGE_HEADING_HINTS:
            return "Usage Examples"
        if heading_norm in STRUCTURE_HEADING_HINTS:
            return "Repository Structure"

        if any(x in heading_norm for x in ["contribut", "pull request", "issue tracker"]):
            return "Contributing Guidelines"

        if any(x in heading_norm for x in ["license", "citation", "terms"]):
            return "License Citation"

        if any(x in heading_norm for x in ["release", "changelog", "version history"]):
            return "Change Log / Release Notes"

        if any(x in heading_norm for x in ["install", "setup", "build"]):
            return "Installation Setup"

        if any(x in heading_norm for x in ["usage", "example", "tutorial", "workflow", "run"]):
            return "Usage Examples"

        if any(x in heading_norm for x in ["structure", "contents", "what's here"]):
            return "Repository Structure"

        if any(x in heading_norm for x in ["config", "configuration", "deployment", "environment"]):
            return "Configuration / Deployment"

    return None

In [106]:
# =========================
# LLM classifier for ambiguous markdown/rst/txt chunks
# =========================

LLM_ALLOWED_TYPES = [
    "Project Overview",
    "Installation Setup",
    "Usage Examples",
    "Repository Structure",
    "Documentation Section",
]

SYSTEM_MSG_CHUNK_CLASSIFIER = (
    "You are a precise classifier for GitHub repository chunks. "
    "Return ONLY valid JSON."
)

def build_chunk_classifier_prompt(
    *,
    repo_name: str,
    source_path: str,
    heading: Optional[str],
    chunk_text: str
) -> str:
    heading_text = heading or ""
    return f"""
# ROLE
Classify a repository chunk into exactly one chunk type.

# REPOSITORY
Repository: {repo_name}
Source path: {source_path}
Section heading: {heading_text}

# CHUNK TEXT
---
{chunk_text}
---

# INSTRUCTIONS
Choose exactly one chunk type from this list:

- Project Overview
- Installation Setup
- Usage Examples
- Repository Structure
- Documentation Section

Use these meanings:
- Project Overview: repo purpose, scope, motivation, architecture, high-level description
- Installation Setup: install, setup, environment preparation, build prerequisites
- Usage Examples: examples, run instructions, workflows, demonstrations, practical execution
- Repository Structure: explanation of folders, files, layout, contents
- Documentation Section: technical or conceptual documentation that does not fit the above

Return exactly one JSON object with this structure:
{{
  "chunk_type": "",
  "reason": ""
}}
""".strip()


def classify_chunk_with_llm(
    *,
    repo_name: str,
    source_path: str,
    heading: Optional[str],
    chunk_text: str
) -> str:
    prompt = build_chunk_classifier_prompt(
        repo_name=repo_name,
        source_path=source_path,
        heading=heading,
        chunk_text=truncate_text(chunk_text, max_chars=8000)
    )

    resp = client.responses.create(
        model=SUMMARY_MODEL,
        input=[
            {"role": "system", "content": SYSTEM_MSG_CHUNK_CLASSIFIER},
            {"role": "user", "content": prompt},
        ],
        text={"format": {"type": "json_object"}, "verbosity": "low"},
        reasoning={"effort": "low"},
    )

    raw = (resp.output_text or "").strip()
    data = json.loads(raw)
    chunk_type = clean_text(data.get("chunk_type"))

    if chunk_type not in LLM_ALLOWED_TYPES:
        return "Documentation Section"

    return chunk_type

In [107]:
# =========================
# Markdown / RST section parsers
# =========================

def parse_markdown_sections(text: str, source_file: str) -> List[Dict[str, Any]]:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = text.split("\n")

    specs = []
    next_local_id = 1
    preamble_lines = []
    current_section = None
    stack = []

    # Tracks whether we are inside a fenced code block delimited by ```
    in_fenced_code_block = False

    def is_fence_line(line: str) -> bool:
        """
        Detect fenced code block delimiters such as:
        ```
        ```bash
        ```python
        """
        stripped = line.lstrip()
        return stripped.startswith("```")

    def clean_section_body(section_lines: List[str]) -> str:
        cleaned = []
        for ln in section_lines:
            if MD_RULE_RE.match(ln):
                continue
            cleaned.append(ln)
        return "\n".join(cleaned).strip()

    def flush_current():
        nonlocal current_section
        if current_section is None:
            return

        body = clean_section_body(current_section["lines"])
        if body or current_section.get("has_children", False):
            if not body:
                body = current_section["title"]

            specs.append({
                "temp_local_id": current_section["temp_local_id"],
                "temp_parent_local_id": current_section["temp_parent_local_id"],
                "chunk_text": truncate_text(body),
                "section_hint": current_section["title"],
                "heading": current_section["title"],
                "heading_level": current_section["heading_level"],
                "source_file": source_file,
                "source_format": "markdown",
            })

        current_section = None

    def create_intro(intro_lines: List[str]):
        nonlocal next_local_id
        intro_text = clean_section_body(intro_lines)
        if not intro_text:
            return

        specs.append({
            "temp_local_id": next_local_id,
            "temp_parent_local_id": None,
            "chunk_text": truncate_text(intro_text),
            "section_hint": "Introduction",
            "heading": "Introduction",
            "heading_level": 0,
            "source_file": source_file,
            "source_format": "markdown",
        })
        next_local_id += 1

    for line in lines:
        # Toggle fenced code block state BEFORE heading detection
        if is_fence_line(line):
            if current_section is None:
                preamble_lines.append(line)
            else:
                current_section["lines"].append(line)

            in_fenced_code_block = not in_fenced_code_block
            continue

        # While inside a fenced code block, never treat lines as markdown headings
        if in_fenced_code_block:
            if current_section is None:
                preamble_lines.append(line)
            else:
                current_section["lines"].append(line)
            continue

        m = MD_HEADING_RE.match(line)
        if m:
            hashes = m.group(1)
            title = m.group(2).strip()
            level = len(hashes)

            if current_section is None and preamble_lines:
                create_intro(preamble_lines)
                preamble_lines = []

            if current_section is not None and level > current_section["heading_level"]:
                current_section["has_children"] = True

            flush_current()

            while stack and stack[-1][0] >= level:
                stack.pop()

            parent_local_id = stack[-1][1] if stack else None

            current_section = {
                "temp_local_id": next_local_id,
                "temp_parent_local_id": parent_local_id,
                "title": title,
                "heading_level": level,
                "lines": [],
                "has_children": False,
            }
            stack.append((level, next_local_id))
            next_local_id += 1

        else:
            if current_section is None:
                preamble_lines.append(line)
            else:
                current_section["lines"].append(line)

    if current_section is None and preamble_lines:
        create_intro(preamble_lines)
    else:
        flush_current()

    return specs


def parse_rst_sections(text: str, source_file: str) -> List[Dict[str, Any]]:
    """
    Lightweight RST parser based on title-underlines.
    """
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = text.split("\n")

    specs = []
    next_local_id = 1

    current_heading = "Introduction"
    current_level = 0
    current_lines = []
    current_parent_local_id = None
    current_temp_id = next_local_id
    next_local_id += 1

    # stack stores tuples: (level, temp_local_id)
    stack = []

    def infer_level(underline: str) -> int:
        ch = underline[0] if underline else "-"
        mapping = {"=": 1, "-": 2, "~": 3, "^": 4, '"': 5, "`": 6}
        return mapping.get(ch, 3)

    def flush_section(
        heading: str,
        level: int,
        lines_block: List[str],
        parent_local_id: Optional[int],
        temp_local_id: int
    ):
        body = "\n".join(lines_block).strip()
        if not body:
            return

        specs.append({
            "temp_local_id": temp_local_id,
            "temp_parent_local_id": parent_local_id,
            "chunk_text": truncate_text(body),
            "section_hint": heading,
            "heading": heading,
            "heading_level": level,
            "source_file": source_file,
            "source_format": "rst",
        })

    i = 0
    while i < len(lines):
        if i + 1 < len(lines):
            title = lines[i].strip()
            underline = lines[i + 1].strip()

            if (
                title
                and underline
                and len(underline) >= len(title)
                and set(underline).issubset(RST_UNDERLINE_CHARS)
            ):
                new_level = infer_level(underline)

                # First flush the previous section using its own stored parent
                flush_section(
                    current_heading,
                    current_level,
                    current_lines,
                    current_parent_local_id,
                    current_temp_id
                )

                # Pop stack until only valid ancestors remain
                while stack and stack[-1][0] >= new_level:
                    stack.pop()

                # Parent is now the top surviving ancestor
                parent_local_id = stack[-1][1] if stack else None

                # Start new current section
                current_heading = title
                current_level = new_level
                current_lines = []
                current_parent_local_id = parent_local_id
                current_temp_id = next_local_id
                next_local_id += 1

                # Push current heading after parent has been determined
                stack.append((new_level, current_temp_id))

                i += 2
                continue

        current_lines.append(lines[i])
        i += 1

    # Flush final section using its stored parent
    flush_section(
        current_heading,
        current_level,
        current_lines,
        current_parent_local_id,
        current_temp_id
    )

    return specs


def parse_rst_sections_old(text: str, source_file: str) -> List[Dict[str, Any]]:
    """
    Lightweight RST parser based on title-underlines.
    """
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = text.split("\n")

    specs = []
    next_local_id = 1
    current_heading = "Introduction"
    current_level = 0
    current_lines = []
    stack = [(0, None)]

    def infer_level(underline: str) -> int:
        ch = underline[0] if underline else "-"
        mapping = {"=": 1, "-": 2, "~": 3, "^": 4, '"': 5, "`": 6}
        return mapping.get(ch, 3)

    def flush_section(heading: str, level: int, lines_block: List[str], parent_local_id: Optional[int], temp_local_id: int):
        body = "\n".join(lines_block).strip()
        if not body:
            return
        specs.append({
            "temp_local_id": temp_local_id,
            "temp_parent_local_id": parent_local_id,
            "chunk_text": truncate_text(body),
            "section_hint": heading,
            "heading": heading,
            "heading_level": level,
            "source_file": source_file,
            "source_format": "rst",
        })

    i = 0
    current_temp_id = next_local_id
    next_local_id += 1

    while i < len(lines):
        if i + 1 < len(lines):
            title = lines[i].strip()
            underline = lines[i + 1].strip()

            if title and underline and len(underline) >= len(title) and set(underline).issubset(RST_UNDERLINE_CHARS):
                parent_local_id = None
                new_level = infer_level(underline)

                while stack and stack[-1][0] >= new_level:
                    stack.pop()
                parent_local_id = stack[-1][1] if stack else None

                flush_section(current_heading, current_level, current_lines, stack[-1][1] if stack else None, current_temp_id)

                current_heading = title
                current_level = new_level
                current_lines = []
                current_temp_id = next_local_id
                next_local_id += 1
                stack.append((new_level, current_temp_id))
                i += 2
                continue

        current_lines.append(lines[i])
        i += 1

    flush_section(current_heading, current_level, current_lines, stack[-1][1] if stack else None, current_temp_id)

    return specs

In [108]:
# =========================
# Notebook parser
# =========================

def notebook_section_hint_from_markdown(md_text: str, fallback: str) -> str:
    lines = md_text.splitlines()
    for ln in lines:
        ln_clean = clean_text(ln)
        if not ln_clean:
            continue
        m = MD_HEADING_RE.match(ln_clean)
        if m:
            return clean_text(m.group(2)) or fallback
        return ln_clean[:120]
    return fallback


def parse_notebook_sections(nb_path: Path, source_file: str) -> List[Dict[str, Any]]:
    nb = read_json(nb_path, {})
    cells = nb.get("cells", []) if isinstance(nb, dict) else []

    specs = []

    for idx, cell in enumerate(cells):
        if not isinstance(cell, dict):
            continue
        if cell.get("cell_type") != "markdown":
            continue

        source = cell.get("source", [])
        if isinstance(source, list):
            md_text = "".join(source)
        else:
            md_text = str(source)

        md_text = clean_multiline_text(md_text)
        if len(md_text) < MIN_TEXT_LENGTH:
            continue

        section_hint = notebook_section_hint_from_markdown(md_text, fallback=f"Notebook cell {idx}")
        specs.append({
            "temp_local_id": None,
            "temp_parent_local_id": None,
            "chunk_text": truncate_text(md_text),
            "section_hint": section_hint,
            "heading": section_hint,
            "heading_level": None,
            "source_file": source_file,
            "source_format": "notebook",
            "extra_type_specific": {
                "notebook_cell_type": "markdown",
                "cell_index_start": idx,
                "cell_index_end": idx,
            }
        })

    return specs

In [109]:
# =========================
# File readers and source-url helpers
# =========================

def infer_source_format(path: str) -> str:
    suffix = Path(path).suffix.lower()
    file_name = Path(path).name.lower()

    if suffix == ".ipynb":
        return "notebook"

    if suffix == ".rst":
        return "rst"

    # Treat common environment / dependency / packaging files as config_text
    if file_name in {
        "requirements",
        "requirements.txt",
        "environment.yml",
        "environment.yaml",
        "pyproject.toml",
        "setup.py",
        "setup.cfg",
        "pipfile",
        "pipfile.lock",
        "dockerfile",
        "docker-compose.yml",
        "docker-compose.yaml",
    }:
        return "config_text"

    if suffix in {".md", ".markdown", ".txt"}:
        return "markdown"

    if suffix in {".yml", ".yaml", ".toml"}:
        return "config_text"

    if suffix == ".py":
        return "python_code"

    return "plain_text"


def read_text_file(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="replace").replace("\x00", "")


def build_source_file_specs(repo_dir: Path, artifact_rec: dict) -> List[Dict[str, Any]]:
    contents_dir = repo_dir / "contents"
    if not contents_dir.exists():
        return []

    specs = []

    for path in sorted([p for p in contents_dir.rglob("*") if p.is_file()], key=lambda p: str(p).lower()):
        rel_path = str(path.relative_to(contents_dir)).replace("\\", "/")
        file_name = path.name
        source_format = infer_source_format(rel_path)

        if source_format == "notebook":
            specs.extend(parse_notebook_sections(path, source_file=rel_path))
            continue

        if source_format == "markdown":
            text = read_text_file(path)
            if len(clean_multiline_text(text)) < MIN_TEXT_LENGTH:
                continue
            specs.extend(parse_markdown_sections(text, source_file=rel_path))
            continue

        if source_format == "rst":
            text = read_text_file(path)
            if len(clean_multiline_text(text)) < MIN_TEXT_LENGTH:
                continue
            specs.extend(parse_rst_sections(text, source_file=rel_path))
            continue

        if source_format == "config_text":
            text = read_text_file(path)
            text = clean_multiline_text(text)
            if len(text) < MIN_TEXT_LENGTH:
                continue

            specs.append({
                "temp_local_id": None,
                "temp_parent_local_id": None,
                "chunk_text": truncate_text(text),
                "section_hint": file_name,
                "heading": file_name,
                "heading_level": None,
                "source_file": rel_path,
                "source_format": source_format,
            })
            continue

        # Keep plain_text only for selected LICENSE-like files
        if source_format == "plain_text":
            text = read_text_file(path)
            text = clean_multiline_text(text)
            if len(text) < MIN_TEXT_LENGTH:
                continue

            if file_name.lower() in {"license", "license.txt", "license.md"}:
                specs.append({
                    "temp_local_id": None,
                    "temp_parent_local_id": None,
                    "chunk_text": truncate_text(text),
                    "section_hint": file_name,
                    "heading": file_name,
                    "heading_level": None,
                    "source_file": rel_path,
                    "source_format": source_format,
                    "forced_chunk_type_name": "License Citation",
                })
            continue

        # Explicitly skip python code files from VectorRAG chunk construction
        if source_format == "python_code":
            continue

    # enrich with source_url
    full_name = artifact_rec.get("full_name")
    frozen_commit_sha = artifact_rec.get("frozen_commit_sha")

    for spec in specs:
        rel_path = spec["source_file"]
        spec.setdefault("extra_type_specific", {})
        spec["extra_type_specific"].update({
            "path": rel_path,
            "source_file": Path(rel_path).name,
            "source_format": spec["source_format"],
            "source_url": build_blob_url(full_name, frozen_commit_sha, rel_path) if full_name and frozen_commit_sha else None,
        })

    return specs

In [110]:
# =========================
# Final chunk-record builder
# =========================

def build_chunk_record(
    *,
    id_artifact: int,
    id_chunk: int,
    order: int,
    chunk_type_name: str,
    chunk_text: str,
    section_hint: str,
    source_artifact_id: int,
    source_file: str,
    supporting_quote: Optional[List[str]] = None,
    id_chunk_parent: Optional[int] = None,
    extra_type_specific: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    return {
        "idArtifact": id_artifact,
        "idChunk": id_chunk,
        "order": order,
        "idChunkType": chunk_type_name_to_id[chunk_type_name],
        "chunk_text": chunk_text,
        "idChunkParent": id_chunk_parent,
        "section_hint": section_hint,
        "supporting_quote": supporting_quote or [],
        "source_artifact_id": source_artifact_id,
        "type_specific": {
            "source_file": Path(source_file).name,
            **(extra_type_specific or {})
        },
    }

In [97]:
# =========================
# Configuration
# =========================

BASE_CODE_REPOS_DIR = Path("./GitHubTest")
ARTIFACTS_PATH = Path("json/coderepo_artifacts.json")
OUTPUT_CHUNKS_PATH = Path("json/coderepo_chunks_test.json")


In [111]:
# =========================
# Build coderepo_chunks.json
# =========================

if not BASE_CODE_REPOS_DIR.exists():
    raise FileNotFoundError(f"Base folder not found: {BASE_CODE_REPOS_DIR}")

repo_dirs = sorted(
    [p for p in BASE_CODE_REPOS_DIR.iterdir() if p.is_dir()],
    key=lambda p: p.name.lower()
)

coderepo_chunks = []
errors = []
next_chunk_id = 1

for repo_dir in repo_dirs:
    repo_name = repo_dir.name
    artifact_rec = repo_name_to_artifact.get(repo_name)

    if artifact_rec is None:
        errors.append({
            "repo_name": repo_name,
            "error": "Missing idArtifact mapping from coderepo_artifacts.json"
        })
        continue

    id_artifact = int(artifact_rec["idArtifact"])

    try:
        file_specs = build_source_file_specs(repo_dir, artifact_rec)

        # Resolve temporary per-file parent ids to real chunk ids
        temp_local_to_real_chunk_id = {}
        projected_chunk_id = next_chunk_id

        for spec in file_specs:
            temp_local_id = spec.get("temp_local_id")
            source_file = spec.get("source_file")
            if temp_local_id is not None and source_file is not None:
                temp_local_to_real_chunk_id[(source_file, temp_local_id)] = projected_chunk_id
            projected_chunk_id += 1

        order_counter = 1

        for spec in file_specs:
            source_path = spec["source_file"]
            source_format = spec["source_format"]
            heading = spec.get("heading")
            chunk_text = spec["chunk_text"]

            # 1. Use forced type first when present
            chunk_type_name = spec.get("forced_chunk_type_name")

            # 2. Otherwise use deterministic classifier
            if chunk_type_name is None:
                chunk_type_name = classify_obvious_chunk_type(
                    source_path=source_path,
                    source_format=source_format,
                    heading=heading,
                )

            # 3. If still unresolved, use LLM classifier
            if chunk_type_name is None:
                chunk_type_name = classify_chunk_with_llm(
                    repo_name=repo_name,
                    source_path=source_path,
                    heading=heading,
                    chunk_text=chunk_text,
                )

            parent_real_id = None
            temp_parent_local_id = spec.get("temp_parent_local_id")
            if temp_parent_local_id is not None:
                parent_real_id = temp_local_to_real_chunk_id.get((source_path, temp_parent_local_id))

            coderepo_chunks.append(
                build_chunk_record(
                    id_artifact=id_artifact,
                    id_chunk=next_chunk_id,
                    order=order_counter,
                    chunk_type_name=chunk_type_name,
                    chunk_text=chunk_text,
                    section_hint=spec["section_hint"],
                    source_artifact_id=id_artifact,
                    source_file=source_path,
                    supporting_quote=[],
                    id_chunk_parent=parent_real_id,
                    extra_type_specific=spec.get("extra_type_specific"),
                )
            )

            next_chunk_id += 1
            order_counter += 1

    except Exception as e:
        errors.append({
            "repo_name": repo_name,
            "error": str(e)
        })

print(f"Chunks built: {len(coderepo_chunks)}")
print(f"Repositories processed: {len(repo_dirs)}")
print(f"Errors: {len(errors)}")

if errors:
    print("Sample errors:")
    for e in errors[:10]:
        print(e)

Chunks built: 2915
Repositories processed: 51
Errors: 0


In [112]:
# =========================
# Save coderepo_chunks.json
# =========================

OUTPUT_CHUNKS_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(coderepo_chunks, f, indent=2, ensure_ascii=False)

print(f"Saved: {OUTPUT_CHUNKS_PATH}")
print(f"Total chunk records: {len(coderepo_chunks)}")

Saved: json/coderepo_chunks.json
Total chunk records: 2915


In [113]:
# =========================
# Quick inspection
# =========================

if coderepo_chunks:
    print(json.dumps(coderepo_chunks[:20], indent=2, ensure_ascii=False)[:20000])

[
  {
    "idArtifact": 1,
    "idChunk": 1,
    "order": 1,
    "idChunkType": 49,
    "chunk_text": "<a href=\"https://colab.research.google.com/gist/KMarkert/d9fb5074fe96e717e9aa7c2e368788cb/nwm_usgs_streamflow_plots.ipynb\" target=\"_parent\"><img src=\"https://colab.research.google.com/assets/colab-badge.svg\" alt=\"Open In Colab\"/></a>",
    "idChunkParent": null,
    "section_hint": "<a href=\"https://colab.research.google.com/gist/KMarkert/d9fb5074fe96e717e9aa7c2e368788cb/nwm_usgs_streamflow_plots.ipyn",
    "supporting_quote": [],
    "source_artifact_id": 1,
    "type_specific": {
      "source_file": "nwm_usgs_streamflow_plot.ipynb",
      "notebook_cell_type": "markdown",
      "cell_index_start": 0,
      "cell_index_end": 0,
      "path": "examples/notebooks/nwm_usgs_streamflow_plot.ipynb",
      "source_format": "notebook",
      "source_url": "https://github.com/CIROH-UA/api-nwm-gcp/blob/3e445c6ba0043b5c2625e75813285603639cd932/examples/notebooks/nwm_usgs_streamflow_pl